# A股原始数据下载

本 Notebook 只下载四份必需的 adata 源缓存，不执行清洗、复权乘数、VWAP、行业中性化或市值计算：

- `adata_listing_dates.parquet`：`all_code()` 股票主表；
- `market_data.parquet`：`k_type=1, adjust_type=2` 后复权 OHLCVA；
- `raw_close.parquet`：`k_type=1, adjust_type=0` 不复权收盘价；
- `stock_shares_history.parquet`：`get_stock_shares(is_history=True)` 历史股本变更记录。

长任务均由你手动运行。下载中断后，重新运行对应单元即可从分段缓存继续。

## 1. 首次环境安装（在 PowerShell 终端执行，不在 Notebook 内执行）

```powershell
$project = 'D:\实习\Gflownet因子挖掘'
& 'D:\Miniconda3\Scripts\conda.exe' create -p "$project\.venv" python=3.12 -y
& "$project\.venv\python.exe" -m pip install --upgrade pip
& "$project\.venv\python.exe" -m pip install -r "$project\requirements.txt"
& "$project\.venv\python.exe" -m ipykernel install --user --name gflownet-factor --display-name 'Python (GFlowNet Factor)'
```

安装后将 Notebook 内核切换为 `Python (GFlowNet Factor)`。

In [ ]:
# 2. 环境与路径
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name.lower() == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import adata
from factor_gfn.data.downloader import (
    DEFAULT_START_DATE,
    LISTING_DATES_PATH,
    MARKET_DATA_PATH,
    RAW_CLOSE_PATH,
    STOCK_SHARES_PATH,
    download_adjusted_market,
    download_raw_close,
    download_stock_shares,
    download_stock_list,
    print_download_summary,
)

print('Python:', sys.version)
print('adata:', getattr(adata, '__version__', 'unknown'))
print('项目目录:', project_root)
print('开始日期:', DEFAULT_START_DATE)
print('股票主表:', LISTING_DATES_PATH)
print('后复权行情:', MARKET_DATA_PATH)
print('不复权收盘价:', RAW_CLOSE_PATH)
print('历史股本:', STOCK_SHARES_PATH)

In [ ]:
# 3. 下载 all_code() 股票主表（通常很快）
# 已有有效缓存时直接读取；确需刷新时改为 force_update=True。
stocks = download_stock_list(force_update=False)
display(stocks.head())
print(stocks['exchange'].value_counts(dropna=False))

In [ ]:
# 4. 断点下载后复权日行情：k_type=1, adjust_type=2
# end_date=None 表示运行当天。中断后原样重跑本单元即可续传。
adjusted_result = download_adjusted_market(
    start_date='2010-01-01',
    end_date=None,
    force_update=False,
)
adjusted_result

In [ ]:
# 5. 断点下载不复权收盘价：k_type=1, adjust_type=0
# 只保存 trade_date、stock_code、close。
raw_close_result = download_raw_close(
    start_date='2010-01-01',
    end_date=None,
    force_update=False,
)
raw_close_result

In [ ]:
# 6. 断点下载历史股本：get_stock_shares(is_history=True)
# 保存总股本、限售股本、流通A股股本和变更原因；中断后原样重跑即可续传。

from factor_gfn.data.downloader import (

    download_stock_shares,
)



shares_result = download_stock_shares(force_update=False)
shares_result

In [ ]:
# 8. 下载结果摘要
# 如果仍有待重试股票，重新运行上面对应的下载单元。
summary = print_download_summary()
summary